# S1-24 `@hobo/conformance/sdk` draft POC — HOBO-614

An SDK idiom layer conforms when it survives five lanes: **golden-IR lowering equivalence**
(bounded by the committed corpus in `src/sdk/corpus/`), the **semantics annex** (committed
vectors for error mapping, cancellation, async ordering, IfC side effects), a **thin-idiom-layer**
scan, an **authorship-neutral gate**, and an honest **coverage claim**. The demonstrations below
run the real gate against the real committed fixtures.

In [1]:
import { strict as assert } from "node:assert";
import { readFileSync } from "node:fs";

import {
  parseLoweringCorpus,
  parseSemanticsAnnexVectors,
  runSdkIdiomConformance,
  sdkConformanceGateSpec,
} from "./src/sdk/index.ts";
import { createDriftAdapter, createReferenceAdapter } from "./test/fixtures/drift-adapters.ts";

const fixture = (name) =>
  JSON.parse(readFileSync(new URL(`./src/sdk/${name}`, import.meta.url), "utf8"));
const corpus = parseLoweringCorpus(fixture("corpus/lowering-corpus.json"));
const annexVectors = parseSemanticsAnnexVectors(fixture("vectors/semantics-annex.vectors.json"));

console.log(`corpus fixtures: ${corpus.map((entry) => entry.name).join(",")}`);
console.log(`annex vectors: ${annexVectors.map((v) => `${v.category}/${v.name}`).join(",")}`);

// The reference adapter passes every lane.
const report = await runSdkIdiomConformance({
  corpus,
  corpusPath: "src/sdk/corpus/lowering-corpus.json",
  annexVectors,
  annexPath: "src/sdk/vectors/semantics-annex.vectors.json",
  adapter: createReferenceAdapter(),
  sourceFiles: [{ path: "src/reference.ts", source: "export const ok = true;" }],
});
for (const lane of report.lanes) console.log(`lane ${lane.lane}: ${lane.status}`);
assert.equal(report.status, "passed");
console.log(`report: suite=${report.suite} contract=${report.contract} status=${report.status}`);

corpus fixtures: orders-handler,bindle-closure,allow-narrowing,secret-ref
annex vectors: error-mapping/result-error-mapping,cancellation/caller-cancellation,async-ordering/awaited-ifc-order,ifc-side-effects/missing-resource-before-handler
lane golden-ir-lowering-equivalence: passed
lane semantics-annex: passed
lane thin-idiom-layer: passed
lane authorship-neutral-gate: passed
lane coverage-claim: passed
report: suite=@hobo/conformance/sdk contract=hobo:sdk@0.1.0 status=passed


## Drift the corpus cannot see, the annex can

A drifted adapter that hides caller cancellation behind a silent retry produces byte-identical
golden-IR lowering — only the semantics-annex vectors expose it, with exact JSON diff paths.
And a helper claim with no backing vector fails the coverage lane: no unearned coverage.

In [2]:
import { strict as assert } from "node:assert";
import { readFileSync } from "node:fs";

import {
  parseLoweringCorpus,
  parseSemanticsAnnexVectors,
  runSdkIdiomConformance,
  sdkConformanceGateSpec,
} from "./src/sdk/index.ts";
import { createDriftAdapter, createReferenceAdapter } from "./test/fixtures/drift-adapters.ts";

const fixture = (name) =>
  JSON.parse(readFileSync(new URL(`./src/sdk/${name}`, import.meta.url), "utf8"));
const corpus = parseLoweringCorpus(fixture("corpus/lowering-corpus.json"));
const annexVectors = parseSemanticsAnnexVectors(fixture("vectors/semantics-annex.vectors.json"));

const drifted = await runSdkIdiomConformance({
  corpus,
  annexVectors,
  adapter: createDriftAdapter({ hideCancellationAsRetry: true }),
});
const goldenIr = drifted.lanes.find((lane) => lane.lane === "golden-ir-lowering-equivalence");
const semantics = drifted.lanes.find((lane) => lane.lane === "semantics-annex");
const cancellation = semantics.cases.find((entry) => entry.name === "caller-cancellation");
assert.equal(goldenIr.status, "passed");
assert.equal(semantics.status, "failed");
console.log(`seeded drift: golden-ir=${goldenIr.status} semantics-annex=${semantics.status}`);
console.log(`failing case=${cancellation.name} diffPaths=${cancellation.diffPaths.join("|")}`);

const uncovered = await runSdkIdiomConformance({
  corpus,
  annexVectors,
  adapter: createReferenceAdapter(),
  helperClaims: ["sdk.retry.jitter"],
});
const coverage = uncovered.lanes.find((lane) => lane.lane === "coverage-claim");
console.log(`unbacked helper claim: report=${uncovered.status} reason="${coverage.cases[0].reason}"`);
console.log("CONTRACT HELD: sdk conformance is corpus-bounded, drift-detecting, and coverage-honest");

seeded drift: golden-ir=passed semantics-annex=failed
failing case=caller-cancellation diffPaths=err|ok
unbacked helper claim: report=failed reason="helper claim has no golden-IR or semantics-annex vector"
CONTRACT HELD: sdk conformance is corpus-bounded, drift-detecting, and coverage-honest
